# 37 · Quem usa relógio integrado ao app é mais engajado?

Três fatores (ativos, engajamento e churn) por uso de relógio.

Duas formas de saber quem tem relógio, cada uma com um limite:
- **`relogio_conectado`** (cadastro): situação na exportação, sem data (decisoes.md pede para não usar
  como preditor de saída). Aqui serve para descrever, porque não depende de a conta ter treinado.
- **Origem dos treinos** (`garmin`, `apple_health`): tem data, mas só enxerga o relógio de quem treinou
  com ele. Quem não treina fica "sem relógio" por construção.

Leitura adotada: grupo pelo cadastro (nenhum, Garmin, Apple Watch), e dentro de quem tem relógio,
se sincronizou treino nos 90 dias antes da data de referência.

Atenção ao medir engajamento: o relógio grava treinos que o app sozinho não gravaria, então parte da
diferença em número de treinos é de registro, não de comportamento.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import NIVEIS

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
REL = ["nenhum", "garmin", "apple_watch"]

usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "relogio_conectado", "sistema"])
assin = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "plano_atual", "inicio", "ativa_no_corte", "renovacao_automatica"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "origem", "duracao_min"])
eng = pd.read_parquet(P / "engajamento_corte.parquet")
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "ativa_inicio_mes", "perdida_no_mes"])
ativ["relogio"] = ativ.origem != "app"
print(len(usuarios), "contas |", len(ativ), "treinos |", int(ativ.relogio.sum()), "de relógio")
print("relógio no cadastro × sistema (todas as contas)"); print(pd.crosstab(usuarios.relogio_conectado, usuarios.sistema, margins=True))
o = ativ.groupby("user_id").origem.agg(lambda s: "+".join(sorted(set(s) - {"app"})) or "só app").rename("origens")
x = usuarios.merge(o, on="user_id", how="left").fillna({"origens": "sem treino em 2026"})
print(); print("relógio no cadastro × origem dos treinos de 2026 (contas)"); print(pd.crosstab(x.relogio_conectado, x.origens, margins=True))

def janela(ref, dias=90):
    a = ativ[(ativ.data <= ref) & (ativ.data > ref - pd.Timedelta(days=dias))]
    g = a.groupby("user_id").agg(treinos_90d=("data", "size"), treinos_relogio=("relogio", "sum"), dias_ativos=("data", "nunique"), n_tipos=("tipo", "nunique"), minutos=("duracao_min", "sum"))
    g["pct_relogio"] = g.treinos_relogio / g.treinos_90d * 100
    return g

63929 contas | 1252488 treinos | 372769 de relógio
relógio no cadastro × sistema (todas as contas)
sistema            android    ios    All
relogio_conectado                       
apple_watch              0  11210  11210
garmin               13204      0  13204
nenhum               21392  18123  39515
All                  34596  29333  63929



relógio no cadastro × origem dos treinos de 2026 (contas)
origens            apple_health  garmin  sem treino em 2026  só app    All
relogio_conectado                                                         
apple_watch                6337       0                4662     211  11210
garmin                        0    7360                5727     117  13204
nenhum                        0       0               16989   22526  39515
All                        6337    7360               27378   22854  63929


## 1. Ativos no corte por relógio

In [2]:
at = assin[assin.ativa_no_corte].merge(usuarios, on="user_id", how="left", validate="one_to_one").merge(eng[["user_id", "nivel", "treinos_30d", "dias_sem_treinar"]], on="user_id", how="left", validate="one_to_one")
at = at.join(janela(CORTE), on="user_id").fillna({"treinos_90d": 0, "treinos_relogio": 0, "dias_ativos": 0, "n_tipos": 0})
print("assinantes ativos:", len(at))
at["uso"] = np.select([at.relogio_conectado == "nenhum", at.treinos_relogio > 0, at.treinos_90d > 0], ["a. sem relógio", "b. relógio, sincronizou em 90d", "c. relógio, treinou só pelo app"], "d. relógio, sem treino em 90d")
t = at.groupby("relogio_conectado").agg(ativos=("user_id", "size"), pct_anual=("plano_atual", lambda s: (s == "anual").mean() * 100)).reindex(REL)
t["% dos ativos"] = t.ativos / t.ativos.sum() * 100
print(t); print()
t2 = at.groupby("uso").agg(ativos=("user_id", "size"), pct_anual=("plano_atual", lambda s: (s == "anual").mean() * 100)); t2["% dos ativos"] = t2.ativos / t2.ativos.sum() * 100; print(t2); print()
print("relógio × sistema entre os ativos"); print(pd.crosstab(at.relogio_conectado, at.sistema, margins=True)); print()
print("% dos treinos de 90 dias que vieram do relógio, entre quem tem relógio e treinou:")
print(at[(at.relogio_conectado != "nenhum") & (at.treinos_90d > 0)].groupby("relogio_conectado").pct_relogio.describe()[["count", "mean", "25%", "50%", "75%"]])

assinantes ativos: 14753
                   ativos  pct_anual  % dos ativos
relogio_conectado                                 
nenhum               9118       57.7          61.8
garmin               2878       58.6          19.5
apple_watch          2757       55.5          18.7

                                 ativos  pct_anual  % dos ativos
uso                                                             
a. sem relógio                     9118       57.7          61.8
b. relógio, sincronizou em 90d     5275       54.2          35.8
c. relógio, treinou só pelo app      79       97.5           0.5
d. relógio, sem treino em 90d       281       99.6           1.9

relógio × sistema entre os ativos
sistema            android   ios    All
relogio_conectado                      
apple_watch              0  2757   2757
garmin                2878     0   2878
nenhum                4597  4521   9118
All                   7475  7278  14753

% dos treinos de 90 dias que vieram do relógio, entre

## 2. Engajamento dos ativos por relógio

In [3]:
def niveis(d, por):
    n = pd.crosstab(d[por], d.nivel).reindex(columns=NIVEIS, fill_value=0)
    t = (n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos"))
    g = d.groupby(por)
    t["treinos 30d"] = g.treinos_30d.mean(); t["treinos 90d"] = g.treinos_90d.mean(); t["dias ativos 90d"] = g.dias_ativos.mean()
    t["nº de tipos"] = g.n_tipos.mean(); t["% renovação desligada"] = g.renovacao_automatica.apply(lambda s: (s == "nao").mean() * 100)
    return t
e1 = niveis(at, "relogio_conectado").reindex(REL); print("== por relógio no cadastro"); print(e1); print()
print("== por uso do relógio"); print(niveis(at, "uso")); print()
for pl in ["mensal", "anual"]:
    print("==", pl); print(niveis(at[at.plano_atual == pl], "relogio_conectado").reindex(REL)); print()
print("== só iOS (Apple Watch e Garmin contra nenhum no mesmo sistema)"); print(niveis(at[at.sistema == "ios"], "relogio_conectado").reindex(REL)); print()
print("== só Android"); print(niveis(at[at.sistema == "android"], "relogio_conectado").reindex(["nenhum", "garmin"]))

== por relógio no cadastro
                   1. alto  2. médio  3. baixo  4. possível churn  ativos  treinos 30d  treinos 90d  dias ativos 90d  nº de tipos  % renovação desligada
relogio_conectado                                                                                                                                       
nenhum                73.0      15.6       2.4                9.0    9118          9.5         25.1             20.2          3.5                   15.1
garmin                69.0      18.3       2.7               10.0    2878          9.2         24.5             19.4          3.5                   16.5
apple_watch           74.2      15.4       2.3                8.1    2757         10.0         26.4             20.9          3.6                   14.4

== por uso do relógio
                                 1. alto  2. médio  3. baixo  4. possível churn  ativos  treinos 30d  treinos 90d  dias ativos 90d  nº de tipos  % renovação desligada
uso               

In [4]:
# Quanto da diferença em treinos é registro: treinos gravados no app por quem tem e por quem não tem relógio
a90 = ativ[(ativ.data <= CORTE) & (ativ.data > CORTE - pd.Timedelta(days=90)) & ativ.user_id.isin(at.user_id)]
app = a90[~a90.relogio].groupby("user_id").size().rename("treinos_app_90d")
at = at.join(app, on="user_id").fillna({"treinos_app_90d": 0})
print("treinos em 90 dias por ativo (média): total, só os gravados no app e os do relógio")
print(at.groupby("relogio_conectado").agg(ativos=("user_id", "size"), total=("treinos_90d", "mean"), pelo_app=("treinos_app_90d", "mean"), pelo_relogio=("treinos_relogio", "mean")).reindex(REL)); print()
tp = a90.merge(usuarios[["user_id", "relogio_conectado"]], on="user_id")
print("tipo dos treinos em 90 dias, por relógio (% da coluna)"); print(pd.crosstab(tp.tipo, tp.relogio_conectado, normalize="columns")[REL] * 100); print()
print("tipo dos treinos por origem (% da coluna)"); print(pd.crosstab(tp.tipo, tp.origem, normalize="columns") * 100); print()
print("duração mediana do treino (min) por origem:", a90.groupby("origem").duracao_min.median().round(1).to_dict())

treinos em 90 dias por ativo (média): total, só os gravados no app e os do relógio
                   ativos  total  pelo_app  pelo_relogio
relogio_conectado                                       
nenhum               9118   25.1      25.1           0.0
garmin               2878   24.5       4.0          20.4
apple_watch          2757   26.4       8.2          18.3



tipo dos treinos em 90 dias, por relógio (% da coluna)
relogio_conectado  nenhum  garmin  apple_watch
tipo                                          
caminhada            16.6    16.9         16.9
corrida              29.3    28.6         30.4
forca                16.3    16.7         15.4
funcional            20.8    21.5         20.6
pedal                 7.5     6.8          7.8
yoga                  9.4     9.5          8.9

tipo dos treinos por origem (% da coluna)
origem     app  apple_health  garmin
tipo                                
caminhada 16.6          16.9    17.1
corrida   29.4          30.4    28.5
forca     16.2          15.4    16.6
funcional 20.9          20.5    21.4
pedal      7.5           7.8     6.8
yoga       9.4           9.0     9.5

duração mediana do treino (min) por origem: {'app': 41.4, 'apple_health': 41.7, 'garmin': 41.3}


## 3. Churn por relógio (abr a set/2026)

In [5]:
partes = []
for mes in pd.date_range("2026-04-01", "2026-09-01", freq="MS"):
    k = painel[(painel.mes == mes) & painel.ativa_inicio_mes].join(janela(mes - pd.Timedelta(days=1)), on="user_id").fillna({"treinos_90d": 0, "treinos_relogio": 0})
    partes.append(k)
pm = pd.concat(partes, ignore_index=True).merge(usuarios, on="user_id", how="left", validate="many_to_one")
print("assinatura-mês de abr a set/2026:", len(pm), "| perdas:", int(pm.perdida_no_mes.sum()))
pm["uso"] = np.select([pm.relogio_conectado == "nenhum", pm.treinos_relogio > 0, pm.treinos_90d > 0], ["a. sem relógio", "b. relógio, sincronizou em 90d", "c. relógio, treinou só pelo app"], "d. relógio, sem treino em 90d")
pm["volume"] = pd.cut(pm.treinos_90d, [-1, 0, 12, 30, 1000], labels=["0 treinos", "1-12", "13-30", "31+"])
pm["periodo"] = np.where(pm.mes.isin(pd.to_datetime(["2026-06-01", "2026-07-01"])), "jun-jul", "outros meses")

def churn(d, por):
    g = d.groupby(por, observed=True).agg(ativas=("user_id", "size"), perdas=("perdida_no_mes", "sum"))
    g["churn %"] = g.perdas / g.ativas * 100
    return g
c1 = churn(pm, ["plano", "relogio_conectado"]); print("== por relógio no cadastro e plano"); print(c1["churn %"].unstack(0).reindex(REL).join(c1.ativas.unstack(0), rsuffix=" (ativas)")); print()
c2 = churn(pm, ["plano", "uso"]); print("== por uso do relógio e plano"); print(c2["churn %"].unstack(0).join(c2.ativas.unstack(0), rsuffix=" (ativas)")); print()
print("== MENSAL por sistema e relógio"); x = churn(pm[pm.plano == "mensal"], ["sistema", "relogio_conectado"]); print(x); print()
print("== MENSAL por sistema, relógio e período (falha de GPS do app em jun-jul no Android)"); print(churn(pm[pm.plano == "mensal"], ["sistema", "relogio_conectado", "periodo"])["churn %"].unstack()); print()
print("== MENSAL só iOS: churn % por volume de treino e relógio"); x = churn(pm[(pm.plano == "mensal") & (pm.sistema == "ios")], ["volume", "relogio_conectado"]); print(x["churn %"].unstack().reindex(columns=REL)); print(x.ativas.unstack().reindex(columns=REL)); print()
print("== ANUAL: churn % por volume de treino e relógio (todas as contas)"); x = churn(pm[pm.plano == "anual"], ["volume", "relogio_conectado"]); print(x["churn %"].unstack().reindex(columns=REL)); print(x.ativas.unstack().reindex(columns=REL))

final = t.join(e1[NIVEIS + ["treinos 30d", "% renovação desligada"]].add_prefix("% ").rename(columns={"% treinos 30d": "treinos 30d", "% % renovação desligada": "% renovação desligada"})).join(c1["churn %"].unstack(0).add_prefix("churn % "))
final.to_csv(S / "relogio_ativos_engajamento_churn.csv", encoding="utf-8-sig")
print(); print(final.T)

assinatura-mês de abr a set/2026: 84868 | perdas: 5542
== por relógio no cadastro e plano
plano              anual  mensal  anual (ativas)  mensal (ativas)
relogio_conectado                                                
nenhum               3.0    11.2           30093            22183
garmin               3.5    11.9            9938             6966
apple_watch          3.4     9.8            8727             6961

== por uso do relógio e plano
plano                            anual  mensal  anual (ativas)  mensal (ativas)
uso                                                                            
a. sem relógio                     3.0    11.2           30093            22183
b. relógio, sincronizou em 90d     2.8    10.9           16784            13895
c. relógio, treinou só pelo app   12.2    13.3             254               15
d. relógio, sem treino em 90d      8.8    17.6            1627               17

== MENSAL por sistema e relógio
                           ativas  p